# GHSOM Toolkits Visualization Gallery

This notebook demonstrates all major visualization capabilities of ghsom-toolkits.

## Setup

First, install the required packages:

```bash
pip install ghsom-py ghsom-toolkits[all]
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ghsom import GHSOM
from ghsom_toolkits.adapters import adapt_model, build_lookup_table

# Set random seed for reproducibility
np.random.seed(42)

# Configure matplotlib
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

## 1. Train a GHSOM Model

Let's create synthetic data and train a GHSOM model.

In [ ]:
# Generate synthetic data with 3 clusters
cluster_centers = [
    np.array([0.2, 0.2, 0.2]),
    np.array([0.8, 0.8, 0.8]),
    np.array([0.5, 0.2, 0.8]),
]

data = []
for center in cluster_centers:
    samples = np.random.randn(100, 3) * 0.1 + center
    data.append(samples)

data = np.vstack(data)
print(f"Data shape: {data.shape}")
print(f"Data range: [{data.min():.2f}, {data.max():.2f}]")

In [ ]:
# Train GHSOM
print("Training GHSOM...")
ghsom = GHSOM(
    input_dataset=data,
    t1=0.5,  # Growth threshold
    t2=0.05,  # Expansion threshold
    learning_rate=0.1,
    gaussian_sigma=1.0
)

result = ghsom.train(epochs_number=50)
print("✓ Training complete!")

# Adapt for ghsom-toolkits
model = adapt_model(result, input_dataset_size=len(data))
lookup = build_lookup_table(model)
print(f"Model has {len(lookup)} nodes")

## 2. Hierarchy Visualizations

### 2.1 Full Hierarchy Tree

In [ ]:
from ghsom_toolkits import visualize_ghsom_hierarchy
from IPython.display import Image

# Create hierarchy visualization
visualize_ghsom_hierarchy(
    node=model,
    lookup_table=lookup,
    filename="hierarchy.png"
)

# Display in notebook
Image("hierarchy.png")

### 2.2 Node Highlighting

In [ ]:
from ghsom_toolkits import visualize_node_position

# Find a node with children
target_node_id = None
for node_id, node in lookup.items():
    if len(node.children) > 0 and node_id != "root":
        target_node_id = node_id
        break

if target_node_id:
    visualize_node_position(
        root_node=model,
        lookup_table=lookup,
        node_id=target_node_id,
        filename="highlighted_node.png",
        plot_descendants=True,
        target_node_color="#FF5733"
    )
    display(Image("highlighted_node.png"))
else:
    print("No suitable node found for highlighting")

## 3. Heatmap Visualizations

### 3.1 Weight Vectors Heatmap

In [ ]:
from ghsom_toolkits.plotting import plot_weight_heatmap

# All neurons
fig = plot_weight_heatmap(model, cmap='viridis')
plt.show()

# Single neuron
fig = plot_weight_heatmap(model, neuron_position=(0, 0), cmap='plasma')
plt.show()

### 3.2 U-Matrix

In [ ]:
from ghsom_toolkits.plotting import plot_umatrix

fig = plot_umatrix(model, cmap='gray_r')
plt.show()

### 3.3 Activation Maps

In [ ]:
from ghsom_toolkits.plotting import plot_activation_map

# Visualize activation for samples from different clusters
sample_indices = [0, 100, 200]  # One from each cluster
fig = plot_activation_map(
    node=model,
    data=data,
    sample_indices=sample_indices,
    cmap='YlOrRd'
)
plt.show()

## 4. Cluster Analysis

### 4.1 Cluster Size Distribution

In [ ]:
from ghsom_toolkits.plotting import plot_cluster_distribution

fig = plot_cluster_distribution(model)
plt.show()

### 4.2 Model Statistics

In [ ]:
# Helper functions for statistics
def count_nodes(node):
    """Count total nodes."""
    count = 1
    for child in node.children:
        count += count_nodes(child)
    return count

def max_depth(node, current=0):
    """Find maximum depth."""
    if not node.children:
        return current
    return max(max_depth(child, current + 1) for child in node.children)

def count_leaves(node):
    """Count leaf nodes (clusters)."""
    if not node.children:
        return 1
    return sum(count_leaves(child) for child in node.children)

# Print statistics
print("=== Model Statistics ===")
print(f"Total nodes: {count_nodes(model)}")
print(f"Maximum depth: {max_depth(model)}")
print(f"Number of clusters (leaves): {count_leaves(model)}")
print(f"Root map shape: {model.rows}x{model.columns}")
print(f"Dataset size: {model.input_dataset_size}")

## 5. Model Comparison

Train and compare multiple models with different hyperparameters.

In [ ]:
from ghsom_toolkits.analysis import compare_models, plot_comparison

# Train multiple models
configs = [
    {'t1': 0.7, 't2': 0.1, 'name': 'Loose'},
    {'t1': 0.5, 't2': 0.05, 'name': 'Medium'},
    {'t1': 0.3, 't2': 0.02, 'name': 'Tight'},
]

models = []
names = []

print("Training multiple models...")
for config in configs:
    print(f"  Training {config['name']}...")
    ghsom = GHSOM(input_dataset=data, t1=config['t1'], t2=config['t2'])
    result = ghsom.train(epochs_number=30)
    models.append(adapt_model(result, input_dataset_size=len(data)))
    names.append(config['name'])

print("✓ All models trained")

In [ ]:
# Compare models
comparison = compare_models(models, data, names, compute_qe=True)
print("\n=== Model Comparison ===")
print(comparison)

In [ ]:
# Visualize comparison (bar chart)
fig = plot_comparison(comparison, plot_type='bar', figsize=(14, 6))
plt.show()

In [ ]:
# Visualize comparison (radar chart)
fig = plot_comparison(comparison, plot_type='radar', figsize=(8, 8))
plt.show()

## 6. Interactive Exploration

### 6.1 Sample Path Tracing

In [ ]:
from ghsom_toolkits.interactive import trace_sample_path

# Trace a sample through the hierarchy
sample_idx = 0
path = trace_sample_path(model, data[sample_idx])

print(f"\nSample {sample_idx} path:")
for step in path:
    print(f"  Level {step['level']}: BMU at {step['bmu_position']}, distance={step['distance']:.4f}")

### 6.2 Neuron Exploration

In [ ]:
from ghsom_toolkits.interactive import explore_neuron

# Explore a specific neuron
neuron_info = explore_neuron(model, neuron_position=(0, 0), data=data)

print("\nNeuron (0,0) information:")
print(f"  Position: {neuron_info['position']}")
print(f"  Weight vector shape: {neuron_info['weights'].shape}")
print(f"  Has child: {neuron_info['has_child']}")
if 'activations' in neuron_info:
    print(f"  Mean activation: {np.mean(neuron_info['activations']):.3f}")

### 6.3 Launch Interactive Dashboard

**Note**: The dashboard requires a separate browser window and won't display inline in Jupyter.

```python
from ghsom_toolkits.interactive import launch_dashboard

# Launch dashboard (opens in browser)
launch_dashboard(model, data=data, port=8050)
# Visit: http://localhost:8050
```

## 7. Generating Reports

Create a comprehensive HTML report.

In [ ]:
from ghsom_toolkits.analysis import generate_report

# Generate report
generate_report(
    node=model,
    data=data,
    output_path="ghsom_report.html",
    model_name="GHSOM Gallery Example"
)

print("✓ Report generated: ghsom_report.html")
print("  Open it in your browser to view")

## Summary

This notebook demonstrated:

1. **Training GHSOM models** with ghsom-py
2. **Hierarchy visualizations** (tree, node highlighting)
3. **Heatmap visualizations** (weights, U-Matrix, activations)
4. **Cluster analysis** (distribution, statistics)
5. **Model comparison** (multiple hyperparameter configurations)
6. **Interactive exploration** (sample tracing, neuron inspection)
7. **Report generation** (comprehensive HTML reports)

## Next Steps

- Explore the [Documentation](https://dadmaan.github.io/ghsom-toolkits/)
- Check out [API Reference](https://dadmaan.github.io/ghsom-toolkits/api/plotting/)
- Try with your own data!

## Resources

- **GitHub**: https://github.com/dadmaan/ghsom-toolkits
- **PyPI**: https://pypi.org/project/ghsom-toolkits/
- **ghsom-py**: https://github.com/dadmaan/ghsom-py